# Notebook 01: Foundations and Environment Setup

Set up the GPU environment, understand when to fine-tune, and establish a baseline with the untuned model.

## 1. Setup and GPU Validation

First, let's install the required packages and verify our GPU is available.

In [1]:
# Install dependencies (uncomment if needed)
# !pip install torch transformers accelerate peft trl bitsandbytes datasets matplotlib python-dotenv

In [1]:
import torch
import os
from pathlib import Path

# GPU validation
assert torch.cuda.is_available(), "CUDA GPU not found! This notebook requires an NVIDIA GPU."

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / (1024 ** 3)

print(f'[OK] GPU detected: {gpu.name}')
print(f'  VRAM: {vram_gb:.1f} GB')
print(f'  CUDA: {torch.version.cuda}')
print(f'  PyTorch: {torch.__version__}')

[OK] GPU detected: NVIDIA GeForce RTX 4070 SUPER
  VRAM: 12.0 GB
  CUDA: 12.1
  PyTorch: 2.5.1


In [9]:
torch.cuda.memory_allocated()/(1024**3)

2.1086349487304688

In [3]:
import transformers
import peft
import trl
import bitsandbytes

print(f'[OK] Libraries loaded')
print(f'  transformers: {transformers.__version__}')
print(f'  peft: {peft.__version__}')
print(f'  trl: {trl.__version__}')

[OK] Libraries loaded
  transformers: 5.2.0
  peft: 0.18.1
  trl: 0.28.0


## 2. When to Fine-Tune vs Prompt vs RAG

We've now seen three approaches to customizing LLM behavior. Here's when to use each:

| Approach | What It Does | Best For | Limitations |
|----------|-------------|----------|-------------|
| **Prompting** | Instructions in system prompt | Quick customization, formatting | Forgets between calls, limited by context window |
| **RAG** (Part 4) | Retrieves external docs | Adding knowledge the model doesn't have | Can only search documents, no actions |
| **Agentic RAG** (Part 5) | RAG + tools + reasoning | Multi-step tasks, real-time data, actions | Still uses generic LLM behavior |
| **Fine-Tuning** (Part 6) | Changes model weights | Domain-specific behavior, tone, format | Needs training data, compute, doesn't add knowledge |

### Key Insight

These approaches are **complementary, not competing**:
- Fine-tuning changes **how** the model responds (style, format, domain expertise)
- RAG adds **what** the model knows (external knowledge)
- Agents add **what** the model can do (tools, actions)

The ultimate stack: **Fine-Tuned Model + Agent Loop + RAG + Tools**

## 3. Fine-Tuning Spectrum

There are different ways to fine-tune a model, with varying compute requirements:

### Full Fine-Tuning
- Updates **all** model parameters
- Requires massive VRAM (e.g., 7B model = ~56 GB in fp16)
- Best quality but impractical on consumer GPUs

### LoRA (Low-Rank Adaptation)
- Freezes original weights, adds small trainable matrices
- Trains only **0.1-1%** of parameters
- Much less VRAM, nearly same quality

### QLoRA (Quantized LoRA) -- What We'll Use
- Loads base model in **4-bit precision** (NF4 quantization)
- Applies LoRA on top of the quantized model
- Enables fine-tuning 7B models on a single consumer GPU

```
Full Fine-Tuning        LoRA                    QLoRA
+------------------+    +------------------+    +------------------+
| All params (fp16)|    | Frozen (fp16)    |    | Frozen (4-bit)   |
| 14 GB for 7B     |    | + LoRA (fp16)    |    | + LoRA (fp16)    |
| ~56 GB VRAM      |    | ~14 GB VRAM      |    | ~6 GB VRAM       |
+------------------+    +------------------+    +------------------+
```

### VRAM Requirements

| Model Size | Full FT (fp16) | LoRA (fp16) | QLoRA (4-bit) |
|-----------|---------------|-------------|---------------|
| 1B        | ~4 GB         | ~4 GB       | ~2 GB         |
| 3B        | ~12 GB        | ~12 GB      | ~4 GB         |
| 7B        | ~56 GB        | ~14 GB      | ~6 GB         |
| 13B       | ~104 GB       | ~28 GB      | ~10 GB        |

## 4. Load a Base Model with 4-bit Quantization

We'll load `meta-llama/Llama-3.2-1B-Instruct` in 4-bit precision using `bitsandbytes`.

This is a small but capable model that runs fast on any modern GPU -- perfect for learning fine-tuning.

> **Note:** Llama 3.2 is a gated model. You need to accept the license at [huggingface.co/meta-llama/Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) and set your `HF_TOKEN` in `.env`.
> 
> **Fallback:** If you don't have access, use `microsoft/Phi-3-mini-4k-instruct` instead (ungated, 3.8B parameters).

In [4]:
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv('HF_TOKEN')

# Choose model -- change this if you don't have Llama access
# MODEL_ID = 'meta-llama/Llama-3.2-1B-Instruct'
MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'  # Fallback (ungated)

print(f'[OK] Using model: {MODEL_ID}')

[OK] Using model: microsoft/Phi-3-mini-4k-instruct


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,               # Load weights in 4-bit
    bnb_4bit_quant_type='nf4',       # NormalFloat4 -- best for LLMs
    bnb_4bit_compute_dtype=torch.float16,  # Compute in fp16
    bnb_4bit_use_double_quant=True,  # Double quantization saves more memory
)

print('[OK] Quantization config ready')
print(f'  Type: NF4 (4-bit NormalFloat)')
print(f'  Compute dtype: float16')
print(f'  Double quantization: enabled')

[OK] Quantization config ready
  Type: NF4 (4-bit NormalFloat)
  Compute dtype: float16
  Double quantization: enabled


In [6]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True
)

# Set pad token if not set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f'[OK] Tokenizer loaded')
print(f'  Vocab size: {tokenizer.vocab_size:,}')
print(f'  Pad token: {tokenizer.pad_token}')

[OK] Tokenizer loaded
  Vocab size: 32,000
  Pad token: <|endoftext|>


In [13]:
tokenizer.encode("What is your return policy")

[1724, 338, 596, 736, 8898]

In [7]:
import transformers
print(transformers.__version__)

5.2.0


In [8]:
# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    attn_implementation="eager"
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
vram_used = torch.cuda.memory_allocated() / (1024 ** 3)

print(f'[OK] Model loaded: {total_params / 1e9:.2f}B parameters')
print(f'  VRAM used: {vram_used:.1f} GB')
print(f'  Quantization: 4-bit NF4')

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

[OK] Model loaded: 2.01B parameters
  VRAM used: 2.1 GB
  Quantization: 4-bit NF4


## 5. Test Base Model on E-commerce Queries

Let's test the **untuned** base model on typical e-commerce customer service questions.

These are the same types of questions from our Part 4 FAQ data. The base model hasn't been trained on our specific e-commerce policies, so we expect generic, non-specific answers.

In [10]:
def generate_response(model, tokenizer, query, max_new_tokens=100):
    # Set up the chat format expected by Phi-3
    messages = [
        {"role": "user", "content": query}
    ]
    
    # 1. Rename to 'inputs' for clarity, and explicitly request a dict
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors='pt',
        add_generation_prompt=True,
        return_dict=True  # Ensure we get a BatchEncoding dict
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,  # 2. Unpack the dictionary here using **
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
        
    # 3. Access the 'input_ids' specifically to check the shape for slicing
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], 
        skip_special_tokens=True
    )
    
    return response

In [11]:
# Test queries -- same types as Part 4 FAQ
test_queries = [
    "What is your return policy?",
    "I want to return order #ORD-12345 because it's defective.",
    "How long does shipping take?",
    "Do you offer international shipping?",
    "Can I change my shipping address after placing an order?"
]

# What we want the fine-tuned model to say (from our FAQ data)
expected_answers = [
    "Our return policy allows you to return products within 30 days of purchase for a full refund, provided they are in their original condition and packaging.",
    "I'm sorry to hear about the defective product. I'll process your return for order #ORD-12345 right away. You'll receive a return shipping label within 24 hours.",
    "Standard shipping takes 3-5 business days, while express shipping takes 1-2 business days.",
    "Yes, we offer international shipping to select countries. Costs are calculated at checkout based on your location.",
    "Please contact our customer support team as soon as possible. We'll update the address if the order has not been shipped yet."
]

print('Testing base model on e-commerce queries...')
print('=' * 70)

baseline_responses = []
for i, query in enumerate(test_queries):
    response = generate_response(model, tokenizer, query)
    baseline_responses.append(response)
    
    print(f'\nQ{i+1}: {query}')
    print(f'\nBase Model: {response}')
    print(f'\nExpected:   {expected_answers[i]}')
    print('-' * 70)

Testing base model on e-commerce queries...

Q1: What is your return policy?

Base Model: As an AI developed by Microsoft, I don't sell products or services, so I don't have a traditional return policy. However, if you're referring to Microsoft's AI services, they are designed to be scalable and adaptable based on your use case. If you encounter any issues, Microsoft provides support to address your concerns directly.

Expected:   Our return policy allows you to return products within 30 days of purchase for a full refund, provided they are in their original condition and packaging.
----------------------------------------------------------------------

Q2: I want to return order #ORD-12345 because it's defective.

Base Model: When a customer wants to return an item due to defects, it's essential to handle the situation with care to maintain customer satisfaction and trust. Here's a step-by串 to effectively manage the return:


1. **Acknowledge the Issue**: Begin by acknowledging the cu

In [12]:
# Save baseline responses for comparison in Notebook 03
import json

baseline_data = [
    {'query': q, 'baseline_response': r, 'expected': e}
    for q, r, e in zip(test_queries, baseline_responses, expected_answers)
]

output_path = Path('../data/baseline_responses.json')
with open(output_path, 'w') as f:
    json.dump(baseline_data, f, indent=2)

print(f'[OK] Baseline responses saved to {output_path}')

[OK] Baseline responses saved to ..\data\baseline_responses.json


## 6. Summary

| Aspect | Status |
|--------|--------|
| GPU | Validated |
| Model | Loaded in 4-bit |
| Baseline | Tested on 5 queries |
| Observation | Generic responses, not e-commerce-specific |

The base model gives reasonable but **generic** answers. It doesn't know our specific policies, doesn't use our brand voice, and doesn't follow our response format.

**Next:** Prepare training data from our e-commerce FAQ and generate synthetic conversations.